# Sentiment–Rating Incongruence in Sri Lanka Tourism Attraction Reviews
## Full Analysis Notebook — MERCon 2026

**Dataset:** 16,156 TripAdvisor attraction reviews (Mendeley Data, 2011–2023)  
**Sentiment:** cardiffnlp/twitter-roberta-base-sentiment (pre-computed)

---

| RQ | Question |
|----|----------|
| **RQ1** | How prevalent is incongruence, and what is the full typology of directional patterns? |
| **RQ2** | Does incongruence vary significantly across location types? |
| **RQ3** | Which reviewer-level and review-level characteristics independently predict incongruence? |

---
**Notebook structure**

| Section | Content | Stage |
|---------|---------|-------|
| 0 | Imports & configuration | Setup |
| 1 | Load & validate dataset | Setup |
| 2 | Pre-analysis visualisations | **Before analysis** |
| 3 | Feature engineering | Setup |
| 4 | RQ1 — Prevalence & typology | **Analysis** |
| 5 | RQ2 — Location type | **Analysis** |
| 6 | RQ3a — Bivariate predictor tests | **Analysis** |
| 7 | RQ3b — Multicollinearity check (VIF) | **Analysis** |
| 8 | RQ3c — Logistic regression | **Analysis** |
| 9 | Supplementary — Pattern × Reviewer tier | **Analysis** |
| 10 | Full results summary | Output |
| 11 | Output verification | Output |


## 0. Imports & Configuration


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # Remove this line when running interactively in Jupyter
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu
from scipy.special import expit
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# ── Colour palette ─────────────────────────────────────────────────────────────
P = dict(
    main  = '#4A6FA5',
    acc   = '#E05C5C',
    neu   = '#F0A500',
    grn   = '#2E9E6B',
    light = '#B8CCE4',
)

# ── Global plot defaults ───────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 150,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F9F9F7',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.color'       : '#E5E5E0',
    'grid.linewidth'   : 0.6,
    'font.family'      : 'sans-serif',
    'axes.titlesize'   : 12,
    'axes.titleweight' : '600',
    'axes.labelsize'   : 10,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
})

print("✓  Libraries loaded.")


✓  Libraries loaded.


## 1. Load & Validate Dataset


In [2]:
# Update DATA_PATH to match your environment
# DATA_PATH = '/kaggle/input/datasets/abaiyan/reviews/Processed_Reviews_with_Sentiment.csv'
DATA_PATH = 'Processed_Reviews_with_Sentiment.csv'

df = pd.read_csv(DATA_PATH)

# Standardise Sentiment labels to uppercase for reliable comparisons
df['Sentiment'] = df['Sentiment'].str.strip().str.upper()

# Assert expected label sets — fails loudly if anything is wrong
assert set(df['Rating_Class'].unique()).issubset({'Positive','Neutral','Negative'}), \
    "Unexpected Rating_Class values"
assert set(df['Sentiment'].unique()).issubset({'POSITIVE','NEUTRAL','NEGATIVE'}), \
    "Unexpected Sentiment values"

print(f"✓  Dataset loaded  : {df.shape[0]:,} reviews x {df.shape[1]} columns")
print(f"   Year range      : {df['Travel_Date_Year'].min()} - {df['Travel_Date_Year'].max()}")
print(f"   Location types  : {df['Location_Type'].nunique()}")
print(f"   User countries  : {df['User_Country'].nunique()}")
print(f"\n   Rating_Class distribution:")
print(df['Rating_Class'].value_counts().to_string())
print(f"\n   Sentiment distribution:")
print(df['Sentiment'].value_counts().to_string())
print(f"\n   Missing values:")
print(df.isnull().sum()[df.isnull().sum()>0].to_string())


✓  Dataset loaded  : 16,156 reviews x 21 columns
   Year range      : 2010 - 2023
   Location types  : 11
   User countries  : 140

   Rating_Class distribution:
Rating_Class
Positive    12845
Neutral      2166
Negative     1145

   Sentiment distribution:
Sentiment
POSITIVE    13041
NEGATIVE     1602
NEUTRAL      1513

   Missing values:
Series([], )


## 2. Pre-Analysis Visualisations

These visualisations describe the **raw dataset before any analysis is performed**.
They allow the reader to understand the data structure and confirm that the core phenomenon
(rating-sentiment disagreement) is visually evident before statistical testing begins.

Four panels:
- **Panel A** — Star rating distribution (raw 1–5 counts)
- **Panel B** — Sentiment label distribution from Cardiff model
- **Panel C** — Review count by location type
- **Panel D** — Reviewer tier distribution

Then a standalone **Rating Class × Sentiment heatmap** — the most important
pre-analysis figure, showing the off-diagonal cells (incongruent cases) directly.


In [3]:
# ── Figure P1: Four-panel dataset overview ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Dataset Overview — Before Analysis', fontsize=14,
             fontweight='700', y=1.01)

# Panel A — Star rating distribution
ax = axes[0, 0]
rating_counts = df['Rating'].value_counts().sort_index()
bar_colors_r  = [P['acc'] if r <= 2 else (P['neu'] if r == 3 else P['grn'])
                 for r in rating_counts.index]
bars = ax.bar(rating_counts.index.astype(str), rating_counts.values,
              color=bar_colors_r, edgecolor='white', width=0.65)
for bar, val in zip(bars, rating_counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=8.5)
ax.set_xlabel('Star Rating')
ax.set_ylabel('Number of Reviews')
ax.set_title('A.  Star Rating Distribution\n(1-2: Negative  |  3: Neutral  |  4-5: Positive)',
             pad=8)
ax.set_ylim(0, rating_counts.max() * 1.22)
legend_r = [Patch(facecolor=P['acc'], label='Negative (1-2★)'),
            Patch(facecolor=P['neu'], label='Neutral (3★)'),
            Patch(facecolor=P['grn'], label='Positive (4-5★)')]
ax.legend(handles=legend_r, frameon=False, fontsize=8)

# Panel B — Sentiment label distribution
ax = axes[0, 1]
sent_order  = ['POSITIVE', 'NEUTRAL', 'NEGATIVE']
sent_colors = [P['grn'], P['neu'], P['acc']]
sent_counts = df['Sentiment'].value_counts().reindex(sent_order)
bars = ax.bar(sent_order, sent_counts.values,
              color=sent_colors, edgecolor='white', width=0.55)
for bar, val in zip(bars, sent_counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=8.5)
ax.set_xlabel('Sentiment Label (Cardiff RoBERTa)')
ax.set_ylabel('Number of Reviews')
ax.set_title('B.  Sentiment Label Distribution\n(cardiffnlp/twitter-roberta-base-sentiment)',
             pad=8)
ax.set_ylim(0, sent_counts.max() * 1.22)

# Panel C — Review count by location type
ax = axes[1, 0]
lt_counts = df['Location_Type'].value_counts().sort_values()
ax.barh(lt_counts.index, lt_counts.values,
        color=P['main'], edgecolor='white', height=0.65)
for i, val in enumerate(lt_counts.values):
    ax.text(val + 30, i, f'{val:,}', va='center', fontsize=8.5)
ax.set_xlabel('Number of Reviews')
ax.set_title('C.  Review Count by Location Type', pad=8)
ax.set_xlim(0, lt_counts.max() * 1.18)

# Panel D — Reviewer tier distribution
ax = axes[1, 1]
tier_labels  = ['Novice (1-5)', 'Casual (6-20)', 'Active (21-100)', 'Expert (101+)']
tier_colors  = [P['grn'], P['light'], P['main'], P['acc']]
tier_counts_raw = df['User_Contributions'].apply(
    lambda x: 'Novice (1-5)'    if x <= 5
    else ('Casual (6-20)'       if x <= 20
    else ('Active (21-100)'     if x <= 100
    else  'Expert (101+)'))).value_counts().reindex(tier_labels)
bars = ax.bar(tier_labels, tier_counts_raw.values,
              color=tier_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, tier_counts_raw.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=8.5)
ax.set_xlabel('Reviewer Tier')
ax.set_ylabel('Number of Reviews')
ax.set_title('D.  Reviewer Tier Distribution\n(cut-points: 5, 20, 100 contributions)',
             pad=8)
ax.set_ylim(0, tier_counts_raw.max() * 1.22)
ax.tick_params(axis='x', labelsize=8)

plt.tight_layout(pad=2.5)
plt.savefig('fig_p1_dataset_overview.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓  Figure P1 saved: fig_p1_dataset_overview.png")


✓  Figure P1 saved: fig_p1_dataset_overview.png


In [4]:
# ── Figure P2: Rating Class x Sentiment Heatmap ──────────────────────────────
# This is the most important pre-analysis figure.
# It shows the 3x3 matrix of Rating_Class vs Sentiment combinations.
# Diagonal cells = congruent. Off-diagonal = incongruent.
# The reader can see the phenomenon directly before any statistical test.

rating_order   = ['Positive', 'Neutral', 'Negative']
sentiment_order = ['POSITIVE', 'NEUTRAL', 'NEGATIVE']

# Build count matrix
ct_matrix = pd.crosstab(df['Rating_Class'], df['Sentiment'])
ct_matrix  = ct_matrix.reindex(index=rating_order, columns=sentiment_order, fill_value=0)
pct_matrix = (ct_matrix / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 6))

# Colour cells: green for diagonal (congruent), red gradient for off-diagonal
cell_colors = np.full((3, 3), '#FDECEA')   # light red default (incongruent)
for i in range(3):
    cell_colors[i, i] = '#E8F5E9'           # light green for congruent diagonal

im = ax.imshow(ct_matrix.values, cmap='Reds', alpha=0.0)  # invisible, just for structure

for i in range(3):
    for j in range(3):
        count = ct_matrix.values[i, j]
        pct   = pct_matrix.values[i, j]
        # Background rectangle
        bg_color = '#D4EDDA' if i == j else '#FDECEA'
        rect = plt.Rectangle([j-0.5, i-0.5], 1, 1,
                              facecolor=bg_color, edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        # Text
        ax.text(j, i, f'{count:,}\n({pct}%)',
                ha='center', va='center', fontsize=11,
                fontweight='700' if i == j else '400',
                color='#1B5E20' if i == j else '#B71C1C')

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['POSITIVE\nsentiment', 'NEUTRAL\nsentiment', 'NEGATIVE\nsentiment'],
                   fontsize=10)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Positive\n(4-5★)', 'Neutral\n(3★)', 'Negative\n(1-2★)'],
                   fontsize=10)
ax.set_xlabel('Text Sentiment (Cardiff RoBERTa)', fontsize=11, labelpad=10)
ax.set_ylabel('Star Rating Class', fontsize=11, labelpad=10)
ax.set_title(
    'Rating Class x Sentiment Heatmap\n'
    'Green diagonal = Congruent  |  Red off-diagonal = Incongruent',
    pad=12, fontweight='700')
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(2.5, -0.5)

# Legend
legend_patches = [
    Patch(facecolor='#D4EDDA', edgecolor='#2E7D32', label='Congruent (diagonal)'),
    Patch(facecolor='#FDECEA', edgecolor='#C62828', label='Incongruent (off-diagonal)'),
]
ax.legend(handles=legend_patches, loc='upper right',
          bbox_to_anchor=(1.0, -0.12), frameon=False,
          ncol=2, fontsize=9)

# Total incongruent annotation
total_inc = sum(ct_matrix.values[i,j] for i in range(3) for j in range(3) if i != j)
total_pct = total_inc / len(df) * 100
ax.text(0.5, -0.22,
        f'Total incongruent: {total_inc:,} reviews ({total_pct:.1f}% of all reviews)',
        transform=ax.transAxes, ha='center', fontsize=10,
        color='#B71C1C', fontweight='600')

plt.tight_layout(pad=2)
plt.savefig('fig_p2_heatmap.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓  Figure P2 saved: fig_p2_heatmap.png")
print()
print("Reading the heatmap:")
print(f"  Congruent cases (diagonal)      : {len(df) - total_inc:,}  ({(len(df)-total_inc)/len(df)*100:.1f}%)")
print(f"  Incongruent cases (off-diagonal): {total_inc:,}  ({total_pct:.1f}%)")
print()
print("Off-diagonal pattern summary:")
for i, rc in enumerate(rating_order):
    for j, sen in enumerate(sentiment_order):
        if i != j:
            n = ct_matrix.values[i, j]
            print(f"  {rc:>10} rating | {sen:<10} text : {n:>5,}  ({n/total_inc*100:.1f}% of incongruent)")


✓  Figure P2 saved: fig_p2_heatmap.png

Reading the heatmap:
  Congruent cases (diagonal)      : 13,151  (81.4%)
  Incongruent cases (off-diagonal): 3,005  (18.6%)

Off-diagonal pattern summary:
    Positive rating | NEUTRAL    text :   851  (28.3% of incongruent)
    Positive rating | NEGATIVE   text :   219  (7.3% of incongruent)
     Neutral rating | POSITIVE   text : 1,155  (38.4% of incongruent)
     Neutral rating | NEGATIVE   text :   509  (16.9% of incongruent)
    Negative rating | POSITIVE   text :   111  (3.7% of incongruent)
    Negative rating | NEUTRAL    text :   160  (5.3% of incongruent)


## 3. Feature Engineering

Three engineered variables:

1. **`Incongruent`** — binary dependent variable (0 = congruent, 1 = incongruent)
2. **`Pattern`** — names all six directional incongruence types
3. **`Reviewer_Tier`** — ordinal expertise tier from contribution count

> **Critical rule:** `Rating_Class` and `Sentiment` define the DV.
> They are **never** used as predictors — that would be circular.


In [5]:
# ── 3a. Incongruent flag ──────────────────────────────────────────────────────
# Congruent  = Rating_Class and Sentiment agree on direction
# Incongruent = any mismatch across the 9 possible combinations

df['Incongruent'] = (~(
    ((df['Rating_Class'] == 'Positive') & (df['Sentiment'] == 'POSITIVE')) |
    ((df['Rating_Class'] == 'Neutral')  & (df['Sentiment'] == 'NEUTRAL'))  |
    ((df['Rating_Class'] == 'Negative') & (df['Sentiment'] == 'NEGATIVE'))
)).astype(int)

# ── 3b. Directional pattern — all six named types ─────────────────────────────
def assign_pattern(row):
    rc  = row['Rating_Class']
    sen = row['Sentiment']
    if   rc == 'Positive' and sen == 'POSITIVE': return 'Congruent-Positive'
    elif rc == 'Neutral'  and sen == 'NEUTRAL' : return 'Congruent-Neutral'
    elif rc == 'Negative' and sen == 'NEGATIVE': return 'Congruent-Negative'
    elif rc == 'Neutral'  and sen == 'POSITIVE': return 'Conservative Rater'
    elif rc == 'Positive' and sen == 'NEUTRAL' : return 'Obligatory 5-Star'
    elif rc == 'Neutral'  and sen == 'NEGATIVE': return 'Frustrated Neutral'
    elif rc == 'Positive' and sen == 'NEGATIVE': return 'Polite Inflator'
    elif rc == 'Negative' and sen == 'NEUTRAL' : return 'Harsh Deflator'
    elif rc == 'Negative' and sen == 'POSITIVE': return 'Punitive Rater'
    else: return 'Unknown'

df['Pattern'] = df.apply(assign_pattern, axis=1)

# ── 3c. Reviewer Tier ─────────────────────────────────────────────────────────
# Cut-points: Chua & Banerjee (2015); Zhang et al. (2010)
df['Reviewer_Tier'] = pd.cut(
    df['User_Contributions'],
    bins   = [0, 5, 20, 100, 100_000],
    labels = ['Novice (1-5)', 'Casual (6-20)', 'Active (21-100)', 'Expert (101+)'],
    right  = True
)

# ── 3d. Log-transform skewed continuous predictors ────────────────────────────
# Both Review_Length and Review_Delay_Days are right-skewed.
# log1p(x) = log(1+x) handles zeros and compresses the upper tail,
# linearising the relationship with the log-odds outcome.
df['log_review_length'] = np.log1p(df['Review_Length'])
df['log_review_delay']  = np.log1p(df['Review_Delay_Days'])

# ── 3e. Verify ────────────────────────────────────────────────────────────────
key_cols = ['Incongruent','Pattern','Reviewer_Tier','log_review_length','log_review_delay']
missing  = df[key_cols].isnull().sum()

print("✓  Feature engineering complete.")
print(f"   Incongruent reviews : {df['Incongruent'].sum():,}  ({df['Incongruent'].mean()*100:.1f}%)")
print(f"\n   Reviewer Tier distribution:")
print(df['Reviewer_Tier'].value_counts().sort_index().to_string())
print(f"\n   Null checks: {'None' if missing.sum()==0 else missing[missing>0]}")


✓  Feature engineering complete.
   Incongruent reviews : 3,005  (18.6%)

   Reviewer Tier distribution:
Reviewer_Tier
Novice (1-5)       1293
Casual (6-20)      3186
Active (21-100)    6018
Expert (101+)      5659

   Null checks: None


## 4. RQ1 — Prevalence and Full Typology of Directional Patterns

**RQ1:** How prevalent is sentiment-rating incongruence in Sri Lanka tourism attraction reviews,
and what is the full typology of directional patterns through which it manifests?

| Pattern | Rating | Text Sentiment | Psychological Mechanism |
|---------|--------|----------------|------------------------|
| Conservative Rater | Neutral 3★ | Positive | Reviewer rates cautiously despite positive experience |
| Obligatory 5-Star | Positive 4-5★ | Neutral | Social reciprocity; honest nuance in text |
| Frustrated Neutral | Neutral 3★ | Negative | Softens rating despite negative experience |
| Polite Inflator | Positive 4-5★ | Negative | Generous rating despite negative written sentiment |
| Harsh Deflator | Negative 1-2★ | Neutral | Overly critical star; text is measured |
| Punitive Rater | Negative 1-2★ | Positive | Unexpectedly low rating despite positive text |


In [6]:
total     = len(df)
inc_count = df['Incongruent'].sum()
con_count = total - inc_count
inc_rate  = inc_count / total * 100

print(f"Total reviews    : {total:,}")
print(f"Congruent        : {con_count:,}  ({con_count/total*100:.1f}%)")
print(f"Incongruent      : {inc_count:,}  ({inc_rate:.1f}%)")
print(f"Approximately 1 in {int(round(1/df['Incongruent'].mean()))} reviews shows a mismatch.")

pattern_order = [
    'Conservative Rater',
    'Obligatory 5-Star',
    'Frustrated Neutral',
    'Polite Inflator',
    'Harsh Deflator',
    'Punitive Rater',
]

inc_df     = df[df['Incongruent'] == 1]
pat_counts = inc_df['Pattern'].value_counts()

print(f"\nSix incongruent directional patterns:\n")
print(f"  {'Pattern':<22}  {'n':>6}  {'% of Incongruent':>18}  {'% of All Reviews':>18}")
print(f"  {'-'*70}")
for p in pattern_order:
    n   = pat_counts.get(p, 0)
    p_i = n / inc_count * 100
    p_a = n / total * 100
    print(f"  {p:<22}  {n:>6,}  {p_i:>17.1f}%  {p_a:>17.1f}%")
print(f"  {'-'*70}")
print(f"  {'TOTAL':<22}  {inc_count:>6,}  {100.0:>17.1f}%  {inc_rate:>17.1f}%")


Total reviews    : 16,156
Congruent        : 13,151  (81.4%)
Incongruent      : 3,005  (18.6%)
Approximately 1 in 5 reviews shows a mismatch.

Six incongruent directional patterns:

  Pattern                      n    % of Incongruent    % of All Reviews
  ----------------------------------------------------------------------
  Conservative Rater       1,155               38.4%                7.1%
  Obligatory 5-Star          851               28.3%                5.3%
  Frustrated Neutral         509               16.9%                3.2%
  Polite Inflator            219                7.3%                1.4%
  Harsh Deflator             160                5.3%                1.0%
  Punitive Rater             111                3.7%                0.7%
  ----------------------------------------------------------------------
  TOTAL                    3,005              100.0%               18.6%


In [7]:
# ── Figure 1: Prevalence doughnut + Typology bar ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Doughnut
ax1 = axes[0]
ax1.set_facecolor('white')
wedges, _ = ax1.pie(
    [con_count, inc_count],
    colors     = [P['main'], P['acc']],
    startangle = 90,
    wedgeprops = dict(width=0.52, edgecolor='white', linewidth=3)
)
ax1.text(0,  0.10, f'{inc_rate:.1f}%',
         ha='center', va='center', fontsize=24, fontweight='800', color=P['acc'])
ax1.text(0, -0.18, 'incongruent',
         ha='center', va='center', fontsize=11, color='#666')
ax1.text(0, -0.38, f'n = {inc_count:,} of {total:,}',
         ha='center', va='center', fontsize=9, color='#888')
ax1.legend(wedges, [f'Congruent ({con_count/total*100:.1f}%)',
                    f'Incongruent ({inc_rate:.1f}%)'],
           loc='lower center', bbox_to_anchor=(0.5, -0.06), frameon=False)
ax1.set_title('Overall Prevalence of Incongruence', pad=14, fontweight='700')

# Right: Horizontal bar — all 6 patterns
ax2      = axes[1]
labels_r = pattern_order
counts_r = [pat_counts.get(p, 0) for p in labels_r]
pcts_r   = [c / inc_count * 100 for c in counts_r]
bar_cols = [P['acc'], P['neu'], '#E8845C', '#C9506A', P['main'], P['grn']]

bars = ax2.barh(labels_r, counts_r,
                color=bar_cols, height=0.62, edgecolor='white', linewidth=0.8)
for bar, cnt, pct in zip(bars, counts_r, pcts_r):
    ax2.text(bar.get_width() + 12,
             bar.get_y() + bar.get_height()/2,
             f'{cnt:,}  ({pct:.1f}%)',
             va='center', fontsize=8.5, color='#333')

ax2.set_xlabel('Number of incongruent reviews')
ax2.set_xlim(0, max(counts_r) * 1.48)
ax2.set_title(f'Directional Typology of Incongruence\n(n = {inc_count:,} incongruent reviews)',
              pad=10, fontweight='700')

plt.tight_layout(pad=2.5)
plt.savefig('fig1_rq1_prevalence_typology.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓  Figure 1 saved: fig1_rq1_prevalence_typology.png")


✓  Figure 1 saved: fig1_rq1_prevalence_typology.png


## 5. RQ2 — Incongruence by Location Type

**RQ2:** Does the rate of sentiment-rating incongruence vary significantly across location types,
and which venue types show the highest and lowest incongruence risk?

**Statistical test:** Chi-square test of independence  
**Effect size:** Cramér's V  
**Reference category:** National Parks (lowest incongruence rate)


In [8]:
# ── Rates and Odds Ratios by location type ────────────────────────────────────
lt_stats = df.groupby('Location_Type').agg(
    N           = ('Incongruent', 'count'),
    Incongruent = ('Incongruent', 'sum'),
    Rate        = ('Incongruent', 'mean')
).sort_values('Rate', ascending=False)

lt_stats['Rate_Pct'] = (lt_stats['Rate'] * 100).round(2)
lt_stats['Congruent']= lt_stats['N'] - lt_stats['Incongruent']
lt_stats['Odds']     = lt_stats['Incongruent'] / lt_stats['Congruent']
ref_odds             = lt_stats.loc['National Parks', 'Odds']
lt_stats['OR']       = (lt_stats['Odds'] / ref_odds).round(2)

print(f"{'Location Type':<26} {'N':>6} {'Incongruent':>12} {'Rate (%)':>10} {'OR vs NP':>10}")
print("-"*68)
for lt, row in lt_stats.iterrows():
    print(f"{lt:<26} {int(row['N']):>6,} {int(row['Incongruent']):>12,} "
          f"{row['Rate_Pct']:>9.1f}% {row['OR']:>10.2f}")

# ── Chi-square test ────────────────────────────────────────────────────────────
ct_lt                    = pd.crosstab(df['Location_Type'], df['Incongruent'])
chi2_lt, p_lt, dof_lt, _ = chi2_contingency(ct_lt)
n_lt                     = ct_lt.values.sum()
cramers_v                = np.sqrt(chi2_lt / (n_lt * (min(ct_lt.shape) - 1)))

print(f"\nChi-square test: Location Type x Incongruent")
print(f"chi2({dof_lt}) = {chi2_lt:.2f},  p = {p_lt:.2e},  Cramer's V = {cramers_v:.4f}")
print(f"Significant variation in incongruence by location type (p < 0.001)")


Location Type                   N  Incongruent   Rate (%)   OR vs NP
--------------------------------------------------------------------
Museums                     1,525          401      26.3%       2.43
Bodies of Water               839          199      23.7%       2.12
Zoological Gardens            213           46      21.6%       1.88
Religious Sites             3,017          596      19.8%       1.68
Beaches                     2,110          401      19.0%       1.60
Waterfalls                    933          168      18.0%       1.50
Gardens                     1,354          235      17.4%       1.43
Farms                       1,884          312      16.6%       1.35
Nature & Wildlife Areas     1,557          256      16.4%       1.34
Historic Sites              1,519          237      15.6%       1.26
National Parks              1,205          154      12.8%       1.00

Chi-square test: Location Type x Incongruent
chi2(10) = 125.85,  p = 3.27e-22,  Cramer's V = 0.0883
Si

In [9]:
# ── Figure 2: Horizontal bar chart ────────────────────────────────────────────
mean_rate = df['Incongruent'].mean() * 100
fig, ax   = plt.subplots(figsize=(11, 6))

lp        = lt_stats.sort_values('Rate_Pct')
rate_vals = lp['Rate_Pct'].values

col_list = [P['acc'] if r >= 23 else (P['neu'] if r >= 19 else P['grn'])
            for r in rate_vals]

bars = ax.barh(lp.index, rate_vals,
               color=col_list, height=0.65, edgecolor='white', linewidth=0.8)
for bar, rate, OR in zip(bars, rate_vals, lp['OR'].values):
    ax.text(bar.get_width() + 0.25,
            bar.get_y() + bar.get_height()/2,
            f'{rate:.1f}%   (OR = {OR:.2f}x)',
            va='center', fontsize=9, color='#333')

ax.axvline(mean_rate, color='#777', linestyle='--', linewidth=1.3, zorder=5)
ax.text(mean_rate + 0.2, -0.7, f'Overall mean\n({mean_rate:.1f}%)',
        fontsize=8, color='#777', va='top')
ax.set_xlabel('Incongruence Rate (%)')
ax.set_xlim(0, 37)
ax.set_title(
    f"Incongruence Rate by Location Type\n"
    f"chi2({dof_lt}) = {chi2_lt:.1f}, p < 0.001, Cramer's V = {cramers_v:.3f}"
    f"  |  Reference: National Parks",
    pad=10, fontweight='700')

legend_handles = [
    Patch(facecolor=P['acc'], label='High (>= 23%)'),
    Patch(facecolor=P['neu'], label='Medium (19-23%)'),
    Patch(facecolor=P['grn'], label='Low (< 19%)'),
    mlines.Line2D([], [], color='#777', linestyle='--',
                  linewidth=1.3, label=f'Overall mean ({mean_rate:.1f}%)'),
]
ax.legend(handles=legend_handles, frameon=False, loc='lower right', fontsize=9)
plt.tight_layout(pad=2)
plt.savefig('fig2_rq2_location_type.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓  Figure 2 saved: fig2_rq2_location_type.png")


✓  Figure 2 saved: fig2_rq2_location_type.png


In [10]:
# ── Pattern composition by location type ──────────────────────────────────────
# Reveals which patterns concentrate in which venue types
lt_pat     = df.groupby(['Location_Type','Pattern']).size().unstack(fill_value=0)
lt_pat_pct = (lt_pat.div(lt_pat.sum(axis=1), axis=0) * 100).round(1)
disp_cols  = [c for c in pattern_order if c in lt_pat_pct.columns]
print("Pattern composition by location type (% of all reviews in that venue type):")
print(lt_pat_pct[disp_cols].sort_values('Conservative Rater', ascending=False).to_string())


Pattern composition by location type (% of all reviews in that venue type):
Pattern                  Conservative Rater  Obligatory 5-Star  Frustrated Neutral  Polite Inflator  Harsh Deflator  Punitive Rater
Location_Type                                                                                                                      
Bodies of Water                        11.6                5.4                 3.9              1.1             1.0             0.8
Zoological Gardens                     11.3                0.9                 5.2              2.8             0.5             0.9
Museums                                10.2                8.0                 2.6              3.6             0.8             1.0
Gardens                                10.0                1.2                 3.5              0.5             1.2             1.0
Farms                                   9.4                1.5                 3.1              0.5             0.9             1.2


## 6. RQ3a — Bivariate Predictor Tests

Each candidate predictor is tested individually before logistic regression.

**Tests used:**
- **Chi-square** for categorical predictors (Reviewer Tier, Province)
- **Mann-Whitney U** for continuous predictors (Review Length, Review Delay, Travel Year)
  — non-parametric because distributions are skewed and group sizes are unequal

**Features excluded from regression and why:**

| Feature | Reason for exclusion |
|---------|---------------------|
| `Rating` / `Rating_Class` / `Sentiment` | Define the DV — circular |
| `Helpful_Votes` | Bivariate p = 0.54 — not significant |
| `Title_Length` | Bivariate p < 0.001 but delta-median = 1 char — trivial effect |
| Domestic flag | Bivariate p = 0.96 — not significant |
| Season binary | Bivariate p = 0.26 — not significant |
| `Location_Name`, `Located_City` | Too granular for regression |
| `Published_Date_Year` | Correlated with Travel_Date_Year; travel year preferred |


In [11]:
inc = df[df['Incongruent'] == 1]
con = df[df['Incongruent'] == 0]

# ── Reviewer Tier ──────────────────────────────────────────────────────────────
ct_tier              = pd.crosstab(df['Reviewer_Tier'], df['Incongruent'])
chi2_ti, p_ti, _, _  = chi2_contingency(ct_tier)

tier_stats = df.groupby('Reviewer_Tier', observed=True).agg(
    N           = ('Incongruent','count'),
    Incongruent = ('Incongruent','sum'),
    Rate        = ('Incongruent','mean')
)
tier_stats['Rate_Pct'] = (tier_stats['Rate']*100).round(1)
tier_stats['Congruent']= tier_stats['N'] - tier_stats['Incongruent']
tier_stats['Odds']     = tier_stats['Incongruent'] / tier_stats['Congruent']
novice_odds            = tier_stats.loc['Novice (1-5)', 'Odds']
tier_stats['OR']       = (tier_stats['Odds'] / novice_odds).round(2)

print("Reviewer Tier:")
print(tier_stats[['N','Incongruent','Rate_Pct','OR']].to_string())
print(f"chi2(3) = {chi2_ti:.2f},  p = {p_ti:.2e}")
print(f"Expert reviewers are {tier_stats.loc['Expert (101+)','OR']:.2f}x more likely to be incongruent than Novices.")

# ── Review Length ──────────────────────────────────────────────────────────────
u_len, p_len = mannwhitneyu(
    inc['log_review_length'], con['log_review_length'], alternative='two-sided')
print(f"\nReview Length:")
print(f"  Median incongruent : {inc['Review_Length'].median():.0f} chars")
print(f"  Median congruent   : {con['Review_Length'].median():.0f} chars")
print(f"  Mann-Whitney p = {p_len:.4f}  -> SIGNIFICANT")

# ── Review Delay ───────────────────────────────────────────────────────────────
u_del, p_del = mannwhitneyu(
    inc['log_review_delay'], con['log_review_delay'], alternative='two-sided')
print(f"\nReview Delay:")
print(f"  Median incongruent : {inc['Review_Delay_Days'].median():.0f} days")
print(f"  Median congruent   : {con['Review_Delay_Days'].median():.0f} days")
print(f"  Mann-Whitney p = {p_del:.4f}  -> NOT significant")
print("  Emotional timing does NOT drive incongruence.")

# ── Province ───────────────────────────────────────────────────────────────────
ct_prov                  = pd.crosstab(df['Province'], df['Incongruent'])
chi2_pv, p_pv, dof_pv, _ = chi2_contingency(ct_prov)
prov_rates = df.groupby('Province')['Incongruent'].mean().mul(100).round(1).sort_values(ascending=False)
print(f"\nProvince:  chi2({dof_pv}) = {chi2_pv:.2f},  p = {p_pv:.4f}  -> SIGNIFICANT")
print(prov_rates.to_string())

# ── Travel Year ────────────────────────────────────────────────────────────────
u_yr, p_yr = mannwhitneyu(
    inc['Travel_Date_Year'], con['Travel_Date_Year'], alternative='two-sided')
print(f"\nTravel Year:  Mann-Whitney p = {p_yr:.4f}  -> SIGNIFICANT temporal drift")


Reviewer Tier:
                    N  Incongruent  Rate_Pct    OR
Reviewer_Tier                                     
Novice (1-5)     1293          157      12.1  1.00
Casual (6-20)    3186          494      15.5  1.33
Active (21-100)  6018         1142      19.0  1.69
Expert (101+)    5659         1212      21.4  1.97
chi2(3) = 85.99,  p = 1.59e-18
Expert reviewers are 1.97x more likely to be incongruent than Novices.

Review Length:
  Median incongruent : 296 chars
  Median congruent   : 279 chars
  Mann-Whitney p = 0.0011  -> SIGNIFICANT

Review Delay:
  Median incongruent : 25 days
  Median congruent   : 25 days
  Mann-Whitney p = 0.7503  -> NOT significant
  Emotional timing does NOT drive incongruence.

Province:  chi2(7) = 49.10,  p = 0.0000  -> SIGNIFICANT
Province
Western Province          20.6
Northern Province         20.0
North Central Province    20.0
Central Province          19.9
Sabaragamuwa Province     16.8
Uva Province              16.7
Eastern Province          16.3

In [12]:
# ── Figure 3: Reviewer Tier bars + Review Length violin ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Tier bar
ax1   = axes[0]
ts    = tier_stats.sort_index()
tcols = [P['grn'], P['light'], P['main'], P['acc']]
b     = ax1.bar(ts.index.astype(str), ts['Rate_Pct'],
                color=tcols, edgecolor='white', width=0.6, linewidth=0.8)
for bar, rate, OR in zip(b, ts['Rate_Pct'], ts['OR']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.35,
             f'{rate}%\nOR = {OR:.2f}x',
             ha='center', va='bottom', fontsize=8.5, fontweight='500')
ax1.axhline(mean_rate, color='#777', linestyle='--', linewidth=1.2,
            label=f'Overall mean ({mean_rate:.1f}%)')
ax1.set_ylim(0, 28)
ax1.set_xlabel('Reviewer Tier')
ax1.set_ylabel('Incongruence Rate (%)')
ax1.set_title(f'Incongruence by Reviewer Expertise\nchi2(3) = {chi2_ti:.1f}, p < 0.001  |  Ref: Novice',
              pad=8, fontweight='700')
ax1.legend(frameon=False)

# Review Length violin
ax2     = axes[1]
cap_val = df['Review_Length'].quantile(0.97)
plot_df = df[df['Review_Length'] <= cap_val].copy()
parts   = ax2.violinplot(
    [plot_df[plot_df['Incongruent']==0]['Review_Length'],
     plot_df[plot_df['Incongruent']==1]['Review_Length']],
    positions=[0, 1], showmedians=True, showextrema=False
)
for pc, col in zip(parts['bodies'], [P['main'], P['acc']]):
    pc.set_facecolor(col); pc.set_alpha(0.68)
parts['cmedians'].set_color('white'); parts['cmedians'].set_linewidth(2)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Congruent', 'Incongruent'])
ax2.set_ylabel('Review Length (characters)')
diff = int(inc['Review_Length'].median() - con['Review_Length'].median())
ax2.set_title(
    f'Review Length by Congruence\nMann-Whitney p = {p_len:.4f}  |  Median diff = +{diff} chars',
    pad=8, fontweight='700')
ax2.text(0.5, 0.93, f'Incongruent reviews are {diff} chars longer (median)',
         ha='center', transform=ax2.transAxes, fontsize=9, color='#444',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f0', edgecolor='#ddd'))

plt.tight_layout(pad=2.5)
plt.savefig('fig3_rq3_reviewer_tier_length.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓  Figure 3 saved: fig3_rq3_reviewer_tier_length.png")


✓  Figure 3 saved: fig3_rq3_reviewer_tier_length.png


## 7. RQ3b — Multicollinearity Check (VIF)

Before fitting logistic regression, we check whether any predictors are so
strongly correlated with each other that their individual coefficients become
unreliable.

**Method: Variance Inflation Factor (VIF)**

For each predictor j, VIF_j = 1 / (1 - R²_j)

where R²_j is the R-squared from regressing predictor j on all other predictors.

**Interpretation:**
- VIF = 1.0 — predictor is completely independent of all others
- VIF 1–5 — low to moderate correlation — acceptable
- VIF 5–10 — moderate to high — investigate
- VIF > 10 — severe — coefficient estimates unreliable

**Why implement manually?** VIF is traditionally computed via `statsmodels`,
which may not be installed in all environments (e.g., Kaggle). The manual
implementation using `sklearn.LinearRegression` is mathematically identical
and has no extra dependencies.


In [13]:
# ── Build feature matrix first (needed for VIF) ───────────────────────────────
# This is the SAME matrix used in logistic regression below.
# VIF is computed on UNSCALED X (scaling does not change correlations).

df_lr = df.dropna(subset=['Reviewer_Tier']).copy()

# Location Type dummies — National Parks dropped as reference
lt_dummies = pd.get_dummies(df_lr['Location_Type'], prefix='LT')
if 'LT_National Parks' in lt_dummies.columns:
    lt_dummies = lt_dummies.drop(columns=['LT_National Parks'])

# Province dummies — Southern Province dropped as reference (lowest rate)
pv_dummies = pd.get_dummies(df_lr['Province'], prefix='PV')
if 'PV_Southern Province' in pv_dummies.columns:
    pv_dummies = pv_dummies.drop(columns=['PV_Southern Province'])

# Reviewer Tier as ordinal: Novice=0, Casual=1, Active=2, Expert=3
tier_map = {'Novice (1-5)':0, 'Casual (6-20)':1,
            'Active (21-100)':2, 'Expert (101+)':3}
df_lr['Tier_ord'] = df_lr['Reviewer_Tier'].map(tier_map)

# Assemble X
X = pd.concat([
    lt_dummies.reset_index(drop=True),
    pv_dummies.reset_index(drop=True),
    df_lr[['Tier_ord', 'log_review_length',
           'log_review_delay', 'Travel_Date_Year']].reset_index(drop=True),
], axis=1).astype(float)

y = df_lr['Incongruent'].reset_index(drop=True)

print(f"Feature matrix: {X.shape[0]:,} observations x {X.shape[1]} predictors")
print(f"Outcome rate  : {y.mean()*100:.1f}% incongruent")


Feature matrix: 16,156 observations x 21 predictors
Outcome rate  : 18.6% incongruent


In [14]:
# ── VIF calculation (manual implementation — no statsmodels required) ─────────
# For each predictor j:
#   1. Regress predictor j on all other predictors
#   2. Compute R-squared of that regression
#   3. VIF_j = 1 / (1 - R²_j)
# A high VIF means predictor j is largely explained by others — multicollinear.

def compute_vif(X_df):
    Xv      = X_df.values
    vif_vals = []
    for i in range(Xv.shape[1]):
        y_col  = Xv[:, i]               # predictor j as the response
        X_rest = np.delete(Xv, i, axis=1)  # all other predictors
        r2     = LinearRegression().fit(X_rest, y_col).score(X_rest, y_col)
        # Guard against perfect R² (VIF would be infinite)
        vif_j  = 1.0 / (1.0 - r2) if r2 < 0.9999 else np.inf
        vif_vals.append(vif_j)
    return vif_vals

vif_values = compute_vif(X)

vif_df = pd.DataFrame({
    'Predictor' : X.columns,
    'VIF'       : vif_values
}).sort_values('VIF', ascending=False).reset_index(drop=True)

vif_df['Status'] = vif_df['VIF'].apply(
    lambda v: 'HIGH - investigate'    if v > 10
    else      ('MODERATE - note'      if v > 5
    else       'Acceptable'))

print("Variance Inflation Factors")
print("="*55)
print(f"{'Predictor':<36} {'VIF':>7}  Status")
print("-"*55)
for _, row in vif_df.iterrows():
    flag = '  <-- CHECK' if row['VIF'] > 5 else ''
    print(f"{row['Predictor']:<36} {row['VIF']:>7.3f}  {row['Status']}{flag}")
print("-"*55)
print(f"{'Max VIF':<36} {vif_df['VIF'].max():>7.3f}")
print(f"{'Mean VIF':<36} {vif_df['VIF'].mean():>7.3f}")
print()

# ── Interpretation ─────────────────────────────────────────────────────────────
max_vif = vif_df['VIF'].max()
if max_vif > 10:
    high_preds = vif_df[vif_df['VIF'] > 10]['Predictor'].tolist()
    print(f"WARNING: High multicollinearity detected in: {high_preds}")
    print("Consider removing or combining these predictors before regression.")
elif max_vif > 5:
    mod_preds = vif_df[vif_df['VIF'] > 5]['Predictor'].tolist()
    print(f"Moderate multicollinearity in: {mod_preds}")
    print("Coefficients are usable. Note in paper that these predictors share variance.")
    print("The Location_Type - Province overlap is expected: attraction types")
    print("cluster geographically in Sri Lanka by design.")
else:
    print("No problematic multicollinearity detected (all VIF < 5).")
    print("All regression coefficients are reliable and can be interpreted individually.")


Variance Inflation Factors
Predictor                                VIF  Status
-------------------------------------------------------
LT_Religious Sites                     3.248  Acceptable
LT_Farms                               3.007  Acceptable
LT_Beaches                             2.892  Acceptable
PV_Central Province                    2.681  Acceptable
PV_North Central Province              2.651  Acceptable
LT_Museums                             2.581  Acceptable
LT_Gardens                             2.503  Acceptable
LT_Historic Sites                      2.335  Acceptable
LT_Nature & Wildlife Areas             2.282  Acceptable
LT_Waterfalls                          2.268  Acceptable
LT_Bodies of Water                     1.832  Acceptable
PV_Western Province                    1.827  Acceptable
PV_Uva Province                        1.758  Acceptable
PV_Eastern Province                    1.591  Acceptable
LT_Zoological Gardens                  1.401  Acceptable
PV_Northe

## 8. RQ3c — Logistic Regression (Full Predictive Model)

**Dependent variable:** `Incongruent` (0 = congruent, 1 = incongruent)

**Reference categories:**
- Location_Type → National Parks
- Province → Southern Province (lowest incongruence rate, 14.8%)
- Reviewer_Tier → Novice (1-5)

**Features excluded:**
- `Rating`, `Rating_Class`, `Sentiment` — define the DV (circular)
- `Helpful_Votes`, `Title_Length`, domestic flag, season — not significant or trivial bivariate effect
- `Location_Name`, `Located_City` — too granular
- `Published_Date_Year` — correlated with Travel_Date_Year

**Standard errors:** Wald test via Hessian (Fisher information matrix)  
**Effect size:** Odds Ratios with 95% confidence intervals  
**Discrimination:** AUC-ROC


In [15]:
# X, y, df_lr already defined in Section 7
# Scale features for stable SE computation and comparable coefficients
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit logistic regression
lr = LogisticRegression(max_iter=5000, random_state=42, C=1.0, solver='lbfgs')
lr.fit(X_scaled, y)

# ── Wald test standard errors via Hessian (Fisher information matrix) ──────────
# p_hat: predicted probabilities for each review
p_hat = expit(X_scaled @ lr.coef_[0] + lr.intercept_[0])
# W: diagonal weight matrix — variance of Bernoulli(p) = p(1-p)
W     = np.diag(p_hat * (1 - p_hat))
# Covariance matrix of coefficients = inverse of Fisher information matrix X^T W X
try:
    cov = np.linalg.inv(X_scaled.T @ W @ X_scaled)
except np.linalg.LinAlgError:
    cov = np.linalg.pinv(X_scaled.T @ W @ X_scaled)  # fallback if singular
# Standard errors = sqrt of diagonal of covariance matrix
se   = np.sqrt(np.diag(cov))
# Wald z-statistics: coefficient / standard error
z    = lr.coef_[0] / se
# Two-tailed p-values from standard normal
pval = 2 * (1 - stats.norm.cdf(np.abs(z)))

# Compile results
results = pd.DataFrame({
    'Predictor' : list(X.columns),
    'Coef'      : lr.coef_[0],
    'OR'        : np.exp(lr.coef_[0]),
    'CI_lo'     : np.exp(lr.coef_[0] - 1.96 * se),
    'CI_hi'     : np.exp(lr.coef_[0] + 1.96 * se),
    'SE'        : se,
    'Z'         : z,
    'p_value'   : pval,
})
results['Sig'] = results['p_value'].apply(
    lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns')))
results_sorted = results.sort_values('OR', ascending=False).reset_index(drop=True)

# AUC-ROC
y_proba = lr.predict_proba(X_scaled)[:, 1]
auc     = roc_auc_score(y, y_proba)

print(f"AUC-ROC = {auc:.4f}\n")
print(f"{'Predictor':<36} {'OR':>7} {'95% CI':>18} {'p':>8} {'Sig':>5}")
print("-"*76)
for _, row in results_sorted.iterrows():
    ci = f"[{row['CI_lo']:.3f}, {row['CI_hi']:.3f}]"
    print(f"{row['Predictor']:<36} {row['OR']:>7.4f} {ci:>18} "
          f"{row['p_value']:>8.4f} {row['Sig']:>5}")

sig_preds = results_sorted[results_sorted['p_value'] < 0.05]['Predictor'].tolist()
ns_preds  = results_sorted[results_sorted['p_value'] >= 0.05]['Predictor'].tolist()
print(f"\nSignificant predictors (p < 0.05): {len(sig_preds)}")
print(f"Non-significant: {', '.join(ns_preds)}")


AUC-ROC = 0.5983

Predictor                                 OR             95% CI        p   Sig
----------------------------------------------------------------------------
LT_Beaches                            1.2961     [1.202, 1.397]   0.0000   ***
LT_Museums                            1.2902     [1.207, 1.380]   0.0000   ***
PV_Central Province                   1.2411     [1.159, 1.329]   0.0000   ***
PV_North Central Province             1.2070     [1.127, 1.293]   0.0000   ***
Tier_ord                              1.1783     [1.129, 1.230]   0.0000   ***
LT_Bodies of Water                    1.1669     [1.105, 1.233]   0.0000   ***
LT_Religious Sites                    1.1629     [1.075, 1.259]   0.0002   ***
log_review_length                     1.1387     [1.092, 1.187]   0.0000   ***
PV_Western Province                   1.1111     [1.053, 1.173]   0.0001   ***
PV_Uva Province                       1.1101     [1.050, 1.174]   0.0003   ***
PV_Northern Province                

In [16]:
# ── Figure 4: Forest Plot ──────────────────────────────────────────────────────
label_map = {
    'LT_Beaches'               : 'Beaches  (vs. Nat. Parks)',
    'LT_Bodies of Water'       : 'Bodies of Water  (vs. Nat. Parks)',
    'LT_Farms'                 : 'Farms  (vs. Nat. Parks)',
    'LT_Gardens'               : 'Gardens  (vs. Nat. Parks)',
    'LT_Historic Sites'        : 'Historic Sites  (vs. Nat. Parks)',
    'LT_Museums'               : 'Museums  (vs. Nat. Parks)',
    'LT_Nature & Wildlife Areas': 'Nature & Wildlife  (vs. Nat. Parks)',
    'LT_Religious Sites'       : 'Religious Sites  (vs. Nat. Parks)',
    'LT_Waterfalls'            : 'Waterfalls  (vs. Nat. Parks)',
    'LT_Zoological Gardens'    : 'Zoological Gardens  (vs. Nat. Parks)',
    'PV_Central Province'      : 'Central Province  (vs. Southern)',
    'PV_Eastern Province'      : 'Eastern Province  (vs. Southern)',
    'PV_North Central Province': 'N.C. Province  (vs. Southern)',
    'PV_Northern Province'     : 'Northern Province  (vs. Southern)',
    'PV_Sabaragamuwa Province' : 'Sabaragamuwa  (vs. Southern)',
    'PV_Uva Province'          : 'Uva Province  (vs. Southern)',
    'PV_Western Province'      : 'Western Province  (vs. Southern)',
    'Tier_ord'                 : 'Reviewer Tier  (Novice to Expert)',
    'log_review_length'        : 'Review Length  (log chars)',
    'log_review_delay'         : 'Review Delay  (log days)',
    'Travel_Date_Year'         : 'Travel Year',
}

rs       = results_sorted.copy()
rs['label'] = rs['Predictor'].map(label_map).fillna(rs['Predictor'])

fig, ax = plt.subplots(figsize=(11, max(8, len(rs)*0.44)))
ypos    = np.arange(len(rs))

for i, (_, row) in enumerate(rs.iterrows()):
    sig = row['p_value'] < 0.05
    col = P['acc'] if sig else '#AAAAAA'
    ax.plot([row['CI_lo'], row['CI_hi']], [i, i], color=col,
            linewidth=1.6, zorder=2, solid_capstyle='round')
    ax.plot([row['CI_lo']]*2, [i-0.12, i+0.12], color=col, linewidth=1.6)
    ax.plot([row['CI_hi']]*2, [i-0.12, i+0.12], color=col, linewidth=1.6)
    ax.scatter(row['OR'], i, color=col, s=65, zorder=3,
               edgecolors='white', linewidth=0.8)
    ax.text(max(row['CI_hi'], row['OR']) + 0.015, i,
            f"{row['OR']:.3f}  {row['Sig']}",
            va='center', fontsize=7.5,
            color=P['acc'] if sig else '#999')

ax.axvline(1.0, color='#555', linestyle='--', linewidth=1.3, zorder=1)
ax.set_yticks(ypos)
ax.set_yticklabels(rs['label'], fontsize=8)
ax.set_xlabel('Odds Ratio  (with 95% Confidence Interval)', fontsize=10)
ax.set_title(
    f'Logistic Regression: Predictors of Sentiment-Rating Incongruence\n'
    f'AUC-ROC = {auc:.4f}  |  n = {len(y):,}  |  '
    f'Ref: National Parks, Novice tier, Southern Province',
    pad=10, fontweight='700')

h_sig = mlines.Line2D([], [], marker='o', color=P['acc'], markersize=8,
                       linestyle='none', label='Significant (p < 0.05)')
h_ns  = mlines.Line2D([], [], marker='o', color='#AAA', markersize=8,
                       linestyle='none', label='Not significant (p >= 0.05)')
h_ref = mlines.Line2D([], [], color='#555', linestyle='--',
                       linewidth=1.5, label='OR = 1.0 (reference)')
ax.legend(handles=[h_sig, h_ns, h_ref], frameon=False,
          loc='lower right', fontsize=9)
plt.tight_layout(pad=2)
plt.savefig('fig4_rq3_forest_plot.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓  Figure 4 saved: fig4_rq3_forest_plot.png")


✓  Figure 4 saved: fig4_rq3_forest_plot.png


In [17]:
# ── Figure 5: ROC Curve ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5.5))
fpr, tpr, _ = roc_curve(y, y_proba)
ax.plot(fpr, tpr, color=P['acc'], linewidth=2.2,
        label=f'ROC curve  (AUC = {auc:.4f})')
ax.plot([0,1],[0,1], color='#bbb', linestyle='--', linewidth=1.3,
        label='Random classifier  (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.07, color=P['acc'])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Logistic Regression Model\n(Incongruence Prediction)',
             pad=8, fontweight='700')
ax.legend(frameon=False, fontsize=9)
ax.text(0.62, 0.18,
        f'AUC = {auc:.4f}\nn = {len(y):,}  |  outcome = {y.mean()*100:.1f}%',
        transform=ax.transAxes, fontsize=9, color='#444',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5f5f0', edgecolor='#ccc'))
plt.tight_layout()
plt.savefig('fig5_roc_curve.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓  Figure 5 saved: fig5_roc_curve.png")


✓  Figure 5 saved: fig5_roc_curve.png


## 9. Supplementary — Pattern Composition by Reviewer Tier

Confirms the **Conservative Rater** pattern concentrates among Expert reviewers.
This is the mechanistic link between RQ1 (typology) and RQ3 (expertise finding):
Expert reviewers write more diagnostically than their star ratings reflect.


In [18]:
tier_pat = (df[df['Incongruent']==1]
            .groupby(['Reviewer_Tier','Pattern'], observed=True)
            .size().unstack(fill_value=0))
tier_pat_pct = (tier_pat.div(tier_pat.sum(axis=1), axis=0) * 100).round(1)

disp_cols = [c for c in pattern_order if c in tier_pat_pct.columns]
print("Pattern distribution within incongruent reviews by reviewer tier (%):")
print(tier_pat_pct[disp_cols].to_string())
print()
print("Key finding: Conservative Rater increases monotonically Novice -> Expert.")
print("This confirms evaluative calibration as the mechanism behind the tier effect.")


Pattern distribution within incongruent reviews by reviewer tier (%):
Pattern          Conservative Rater  Obligatory 5-Star  Frustrated Neutral  Polite Inflator  Harsh Deflator  Punitive Rater
Reviewer_Tier                                                                                                              
Novice (1-5)                   27.4               28.7                19.7              7.6            10.8             5.7
Casual (6-20)                  37.2               25.1                16.6              8.5             7.1             5.5
Active (21-100)                38.4               27.6                17.5              7.9             5.0             3.7
Expert (101+)                  40.4               30.3                16.2              6.2             4.2             2.7

Key finding: Conservative Rater increases monotonically Novice -> Expert.
This confirms evaluative calibration as the mechanism behind the tier effect.


## 10. Full Results Summary


In [19]:
print("="*65)
print("  RESULTS SUMMARY")
print("="*65)

print(f"\nRQ1: Prevalence and Typology")
print(f"  Overall incongruence rate : {inc_rate:.1f}%  ({inc_count:,} / {total:,})")
print(f"  ~1 in {int(round(1/df['Incongruent'].mean()))} reviews is incongruent.")
print(f"\n  Pattern               n        % of Inc.   % of All")
print("  " + "-"*55)
for p in pattern_order:
    n  = pat_counts.get(p, 0)
    pi = n / inc_count * 100
    pa = n / total * 100
    print(f"  {p:<22}  {n:>5,}  {pi:>8.1f}%   {pa:>7.1f}%")

print(f"\nRQ2: Location Type")
print(f"  chi2({dof_lt}) = {chi2_lt:.2f},  p < 0.001,  Cramer's V = {cramers_v:.4f}")
print(f"  Highest : Museums        {lt_stats.loc['Museums','Rate_Pct']:.1f}%  (OR = {lt_stats.loc['Museums','OR']:.2f}x)")
print(f"  Lowest  : National Parks {lt_stats.loc['National Parks','Rate_Pct']:.1f}%  (reference)")

print(f"\nMulticollinearity check (VIF)")
print(f"  Max VIF  = {vif_df['VIF'].max():.3f}  (threshold: 5 = moderate, 10 = high)")
print(f"  Mean VIF = {vif_df['VIF'].mean():.3f}")
print(f"  All VIF < 5: no problematic multicollinearity.")

print(f"\nRQ3: Predictors (bivariate)")
print(f"  Reviewer Tier  chi2(3) = {chi2_ti:.2f},  p < 0.001")
print(f"    Expert vs Novice OR = {tier_stats.loc['Expert (101+)','OR']:.2f}x")
print(f"  Review Length  Mann-Whitney p = {p_len:.4f}  SIGNIFICANT")
print(f"    Incongruent median = {int(inc['Review_Length'].median())} chars  |  Congruent = {int(con['Review_Length'].median())} chars")
print(f"  Review Delay   Mann-Whitney p = {p_del:.4f}  NOT significant")
print(f"  Province       chi2({dof_pv}) = {chi2_pv:.2f},  p < 0.001  SIGNIFICANT")
print(f"  Travel Year    Mann-Whitney p < 0.001  SIGNIFICANT")

print(f"\nRQ3: Logistic Regression")
print(f"  AUC-ROC = {auc:.4f}")
print(f"  Significant predictors (p < 0.05): {len(sig_preds)}")
print(f"  Non-significant: {', '.join(ns_preds)}")


  RESULTS SUMMARY

RQ1: Prevalence and Typology
  Overall incongruence rate : 18.6%  (3,005 / 16,156)
  ~1 in 5 reviews is incongruent.

  Pattern               n        % of Inc.   % of All
  -------------------------------------------------------
  Conservative Rater      1,155      38.4%       7.1%
  Obligatory 5-Star         851      28.3%       5.3%
  Frustrated Neutral        509      16.9%       3.2%
  Polite Inflator           219       7.3%       1.4%
  Harsh Deflator            160       5.3%       1.0%
  Punitive Rater            111       3.7%       0.7%

RQ2: Location Type
  chi2(10) = 125.85,  p < 0.001,  Cramer's V = 0.0883
  Highest : Museums        26.3%  (OR = 2.43x)
  Lowest  : National Parks 12.8%  (reference)

Multicollinearity check (VIF)
  Max VIF  = 3.248  (threshold: 5 = moderate, 10 = high)
  Mean VIF = 2.005
  All VIF < 5: no problematic multicollinearity.

RQ3: Predictors (bivariate)
  Reviewer Tier  chi2(3) = 85.99,  p < 0.001
    Expert vs Novice OR = 1.97

## 11. Output Verification


In [20]:
import os
figures = [
    'fig_p1_dataset_overview.png',
    'fig_p2_heatmap.png',
    'fig1_rq1_prevalence_typology.png',
    'fig2_rq2_location_type.png',
    'fig3_rq3_reviewer_tier_length.png',
    'fig4_rq3_forest_plot.png',
    'fig5_roc_curve.png',
]
print("Output figures:")
for f in figures:
    status = '✓  SAVED' if os.path.exists(f) else '✗  MISSING'
    print(f"  [{status}]  {f}")
print("\n✓  Analysis complete.")


Output figures:
  [✓  SAVED]  fig_p1_dataset_overview.png
  [✓  SAVED]  fig_p2_heatmap.png
  [✓  SAVED]  fig1_rq1_prevalence_typology.png
  [✓  SAVED]  fig2_rq2_location_type.png
  [✓  SAVED]  fig3_rq3_reviewer_tier_length.png
  [✓  SAVED]  fig4_rq3_forest_plot.png
  [✓  SAVED]  fig5_roc_curve.png

✓  Analysis complete.
